# Figure 2 — Trial structure of the cued-target task

Publication-quality rendering of one representative trial from the stage-2
training environment (`CuedTargetWithDistractorsV3`, `continuous_locations=True`,
`p_distractor_trial=0.6`).

The figure mirrors what the network actually sees at training time — the
environment instance is the source of truth for epoch boundaries, CTOA, response
window, and distractor placement (nothing is hand-rolled).


In [ ]:
import sys, os, pathlib
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

from src.env import CuedTargetWithDistractorsV3
from src.figsave import autosave

autosave("00_figure_trial_structure", dpi=300)


In [ ]:
plt.rcParams.update({
    "font.family":        "serif",
    "mathtext.fontset":   "cm",
    "font.size":          9,
    "axes.labelsize":     9,
    "axes.titlesize":     10,
    "xtick.labelsize":    8,
    "ytick.labelsize":    8,
    "legend.fontsize":    7.5,
    "axes.linewidth":     0.8,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "xtick.direction":    "out",
    "ytick.direction":    "out",
    "xtick.major.size":   3,
    "ytick.major.size":   3,
    "xtick.major.width":  0.8,
    "ytick.major.width":  0.8,
    "pdf.fonttype":       42,
    "ps.fonttype":        42,
})


## Sample a representative trial

Stage-2 env exactly as in [01c_train_topo.ipynb](01c_train_topo.ipynb).
Rejection-sample seeds for a trial with a medium-long CTOA (~2000 ms) and
exactly one distractor in the middle of the delay — so the figure annotates
both the response window and a representative distractor event.


In [ ]:
def build_env():
    return CuedTargetWithDistractorsV3(
        dt=20,
        cue_strength=1.0,
        p_distractor_trial=0.6,
        distractor_strength=1.0,
        continuous_locations=True,
    )

def trial_is_representative(env, tr):
    if tr["n_distractors"] != 1:
        return False
    if not (1800 <= tr["ctoa_ms"] <= 2400):
        return False
    delay_lo = env.start_ind["delay"]
    delay_hi = env.end_ind["delay"]
    d_onset = tr["distractor_onsets_steps"][0]
    frac = (d_onset - delay_lo) / max(1, (delay_hi - delay_lo))
    return 0.35 <= frac <= 0.65

picked_seed = None
env = None
trial = None
for seed in range(20000):
    env_try = build_env()
    np.random.seed(seed)
    env_try.rng = np.random.RandomState(seed)
    env_try.reset()
    tr = env_try.trial
    if trial_is_representative(env_try, tr):
        picked_seed = seed
        env = env_try
        trial = tr
        break

assert env is not None, "rejection sampling failed; widen the criteria"
print(f"seed = {picked_seed}")
print(f"CTOA = {trial['ctoa_ms']} ms  (sampled from {env.ctoa_range} ms, Beta{env.ctoa_beta})")
print(f"cue_theta  = {math.degrees(trial['cue_theta']):+.1f} deg")
print(f"cue_pos    = ({trial['cue_pos'][0]:+.2f}, {trial['cue_pos'][1]:+.2f})")
print(f"target_pos = ({trial['target_pos'][0]:+.2f}, {trial['target_pos'][1]:+.2f})")
print(f"distractor onset (ms from cue) = {trial['distractor_cdoa_ms'][0]}")
print(f"distractor_pos = ({trial['distractor_positions'][0][0]:+.2f}, "
      f"{trial['distractor_positions'][0][1]:+.2f})")


In [ ]:
ob   = env.ob.copy()
T    = ob.shape[0]
dt   = env.dt
t_ms = np.arange(T) * dt

# Epoch boundaries (ms), pulled from the env — no recomputation
EPOCHS = ["fixation", "cue", "delay", "target", "post_target"]
epoch_ms = {n: (env.start_ind[n] * dt, env.end_ind[n] * dt) for n in EPOCHS}

# Distractor event
d_ev = env._distractor_events[0]
d_on_ms  = d_ev["onset_step"]  * dt
d_off_ms = d_ev["offset_step"] * dt
fa_on_ms = d_ev["fa_start"]    * dt
fa_off_ms= d_ev["fa_end"]      * dt

# Target & response window
t_target0_ms = env.start_ind["target"] * dt
resp_lo_ms, resp_hi_ms = env.rt_window
resp_on_ms  = t_target0_ms + resp_lo_ms
resp_off_ms = t_target0_ms + resp_hi_ms

print(f"total trial length:  {t_ms[-1] + dt:.0f} ms  ({T} timesteps)")
print(f"response window:     {resp_on_ms:.0f}-{resp_off_ms:.0f} ms")
print(f"distractor pulse:    {d_on_ms:.0f}-{d_off_ms:.0f} ms")


## Render the figure

Layout (top→bottom):

1. **Epoch timeline** — five labeled segments, CTOA bracket below.
2. **Fixation channel** (ch 0).
3. **Cue channels** (ch 1 `cue_x`, ch 2 `cue_y`, ch 3 `cue_strength`).
4. **Stimulus channels** (ch 4 `stim_x`, ch 5 `stim_y`, ch 6 `stim_strength`)
   — carry both the target pulse and the distractor pulse.

Vertical shading marks (a) the response window, (b) the target pulse, and
(c) the distractor pulse — extending across all three channel panels.


In [ ]:
# --- palette -----------------------------------------------------------
EPOCH_FILL = {
    "fixation":    "#eceff1",
    "cue":         "#ffe0b2",
    "delay":       "#e3f2fd",
    "target":      "#c8e6c9",
    "post_target": "#f3e5f5",
}
EPOCH_LABEL = {
    "fixation":    "Fixation",
    "cue":         "Cue",
    "delay":       "Delay (CTOA)",
    "target":      "Target",
    "post_target": "Post-target",
}

C_FIX    = "#1f1f1f"
C_X      = "#0072B2"   # Wong blue
C_Y      = "#D55E00"   # Wong vermilion
C_STR    = "#009E73"   # Wong green
C_TARGET = "#1b7837"   # forest green
C_DISTR  = "#b2182b"   # brick red
C_RESP   = "#fdd835"   # amber

def shade_events(ax):
    ax.axvspan(resp_on_ms, resp_off_ms, color=C_RESP,   alpha=0.30, lw=0, zorder=0)
    ax.axvspan(epoch_ms["target"][0], epoch_ms["target"][1],
               color=C_TARGET, alpha=0.20, lw=0, zorder=0)
    ax.axvspan(d_on_ms,    d_off_ms,    color=C_DISTR,  alpha=0.20, lw=0, zorder=0)

DSTYLE = "steps-post"

# --- figure ------------------------------------------------------------
fig = plt.figure(figsize=(7.6, 4.9))
gs = GridSpec(
    nrows=4, ncols=1,
    height_ratios=[0.90, 0.45, 1.05, 1.05],
    hspace=0.18,
    figure=fig,
)
ax_tl   = fig.add_subplot(gs[0])
ax_fix  = fig.add_subplot(gs[1], sharex=ax_tl)
ax_cue  = fig.add_subplot(gs[2], sharex=ax_tl)
ax_stim = fig.add_subplot(gs[3], sharex=ax_tl)

# --- row 0: epoch timeline --------------------------------------------
ax_tl.set_xlim(0, t_ms[-1] + dt)
ax_tl.set_ylim(0, 1)
ax_tl.set_yticks([])
for s in ("left", "right", "top", "bottom"):
    ax_tl.spines[s].set_visible(False)
ax_tl.tick_params(left=False, bottom=False, labelbottom=False)

# Strip in the lower half: above it sit distractor + Target labels,
# below it the CTOA bracket — no dead band anywhere in the row.
bar_lo, bar_hi = 0.34, 0.60
for name in EPOCHS:
    t0, t1 = epoch_ms[name]
    ax_tl.add_patch(mpatches.Rectangle(
        (t0, bar_lo), t1 - t0, bar_hi - bar_lo,
        facecolor=EPOCH_FILL[name], edgecolor="0.25", linewidth=0.6,
    ))

inside_y = (bar_lo + bar_hi) / 2
for name in ("fixation", "cue", "delay", "post_target"):
    t0, t1 = epoch_ms[name]
    ax_tl.text((t0 + t1) / 2, inside_y, EPOCH_LABEL[name],
               ha="center", va="center", fontsize=8.5)

# "Target" leader-line label
tgt_t0, tgt_t1 = epoch_ms["target"]
tgt_mid = 0.5 * (tgt_t0 + tgt_t1)
ax_tl.annotate(
    "Target", xy=(tgt_mid, bar_hi), xytext=(tgt_mid, 0.72),
    ha="center", va="bottom", fontsize=8.5,
    arrowprops=dict(arrowstyle="-", color="0.25", lw=0.6, shrinkA=0, shrinkB=1),
)

# distractor marker above the strip
ax_tl.annotate(
    "distractor",
    xy=(0.5 * (d_on_ms + d_off_ms), bar_hi + 0.02),
    xytext=(0.5 * (d_on_ms + d_off_ms), 0.92),
    ha="center", va="bottom", fontsize=8, color=C_DISTR,
    arrowprops=dict(arrowstyle="-|>", color=C_DISTR, lw=0.8,
                    shrinkA=0, shrinkB=1),
)

# --- BELOW the strip ---------------------------------------------------
# Row A (closer to strip): CTOA under the delay segment.
# Row B (lower): response window under post-target, on its own y-level so
#                the long labels don't collide.
delay_t0, delay_t1 = epoch_ms["delay"]

ctoa_bracket_y = 0.22
ctoa_label_y   = 0.08
ax_tl.annotate(
    "", xy=(delay_t0, ctoa_bracket_y), xytext=(delay_t1, ctoa_bracket_y),
    arrowprops=dict(arrowstyle="<->", linewidth=0.8, color="0.20"),
)
ax_tl.text((delay_t0 + delay_t1) / 2, ctoa_label_y,
           f"CTOA = {trial['ctoa_ms']} ms   "
           f"(sampled from {env.ctoa_range[0]}–{env.ctoa_range[1]} ms)",
           ha="center", va="center", fontsize=8)


# --- row 1: fixation ---------------------------------------------------
shade_events(ax_fix)
ax_fix.plot(t_ms, ob[:, 0], color=C_FIX, lw=1.3, drawstyle=DSTYLE)
ax_fix.set_ylim(-0.18, 1.30)
ax_fix.set_yticks([0, 1])
ax_fix.set_ylabel("fixation\n(ch 0)", rotation=0, ha="right", va="center",
                  labelpad=14, fontsize=8.5)
ax_fix.tick_params(labelbottom=False)
ax_fix.axhline(0, color="0.88", lw=0.5, zorder=0)

# --- row 2: cue group --------------------------------------------------
# Channel numbers now live in the y-axis label, not in the legend.
shade_events(ax_cue)
ax_cue.plot(t_ms, ob[:, 1], color=C_X,   lw=1.1,
            drawstyle=DSTYLE, label=r"$\mathrm{cue}_x$")
ax_cue.plot(t_ms, ob[:, 2], color=C_Y,   lw=1.1,
            drawstyle=DSTYLE, label=r"$\mathrm{cue}_y$")
ax_cue.plot(t_ms, ob[:, 3], color=C_STR, lw=1.6,
            drawstyle=DSTYLE, label="cue strength")
ax_cue.set_ylim(-1.15, 1.30)
ax_cue.set_yticks([-1, 0, 1])
ax_cue.set_ylabel("cue\n(ch 1\u20133)", rotation=0,
                  ha="right", va="center", labelpad=14, fontsize=8.5)
ax_cue.legend(loc="lower left", frameon=True, fancybox=False,
              framealpha=0.95, edgecolor="0.85", ncol=3,
              handlelength=1.4, columnspacing=1.0, borderpad=0.3)
ax_cue.tick_params(labelbottom=False)
ax_cue.axhline(0, color="0.88", lw=0.5, zorder=0)

# --- row 3: stim group -------------------------------------------------
shade_events(ax_stim)
ax_stim.plot(t_ms, ob[:, 4], color=C_X,   lw=1.1,
             drawstyle=DSTYLE, label=r"$\mathrm{stim}_x$")
ax_stim.plot(t_ms, ob[:, 5], color=C_Y,   lw=1.1,
             drawstyle=DSTYLE, label=r"$\mathrm{stim}_y$")
ax_stim.plot(t_ms, ob[:, 6], color=C_STR, lw=1.6,
             drawstyle=DSTYLE, label="stim strength")
ax_stim.set_ylim(-1.15, 1.30)
ax_stim.set_yticks([-1, 0, 1])
ax_stim.set_ylabel("stimulus\n(ch 4\u20136)", rotation=0,
                   ha="right", va="center", labelpad=14, fontsize=8.5)
ax_stim.legend(loc="lower left", frameon=True, fancybox=False,
               framealpha=0.95, edgecolor="0.85", ncol=3,
               handlelength=1.4, columnspacing=1.0, borderpad=0.3)
ax_stim.axhline(0, color="0.88", lw=0.5, zorder=0)
ax_stim.set_xlabel("time from trial onset (ms)")

xmax = t_ms[-1] + dt
ax_stim.set_xticks(np.arange(0, xmax + 1, 500))

# --- shared event legend below x label --------------------------------
event_handles = [
    mpatches.Patch(facecolor=C_RESP,   alpha=0.55, edgecolor="none",
                   label="response window"),
    mpatches.Patch(facecolor=C_TARGET, alpha=0.45, edgecolor="none",
                   label=f"target pulse ({env.timing['target']} ms)"),
    mpatches.Patch(facecolor=C_DISTR,  alpha=0.45, edgecolor="none",
                   label=f"distractor pulse ({env.distractor_duration} ms)"),
]
fig.legend(handles=event_handles, loc="lower center", ncol=3,
           frameon=False, fontsize=8, bbox_to_anchor=(0.5, -0.03))

fig.suptitle("Trial structure of the cued-target task (stage-2 environment)",
             fontsize=10.5, y=0.985)

out_dir = Path("figures"); out_dir.mkdir(exist_ok=True)
fig.savefig(out_dir / "figure2_trial_structure.pdf", bbox_inches="tight")
fig.savefig(out_dir / "figure2_trial_structure.png", dpi=300, bbox_inches="tight")
print("saved figures/figure2_trial_structure.{pdf,png}")

plt.show()


## Sanity checks against the environment instance

These prints reproduce the values used in the figure straight from the env so
that any future change in defaults is caught here, not silently in the figure.


In [ ]:
print("dt:                  ", env.dt, "ms")
print("fixation_jitter:     ", env.fixation_jitter, "ms")
print("cue_duration:        ", env.cue_duration, "ms")
print("ctoa_range:          ", env.ctoa_range, "ms")
print("target duration:     ", env.timing["target"], "ms")
print("post_target_duration:", env.post_target_duration, "ms")
print("rt_window:           ", env.rt_window, "ms post-target")
print("distractor_duration: ", env.distractor_duration, "ms")
print("p_distractor_trial:  ", env.p_distractor_trial)
print()
print("epoch durations of THIS trial (ms):")
for n in EPOCHS:
    t0, t1 = epoch_ms[n]
    print(f"  {n:11s}  {t0:6.0f} - {t1:6.0f}   ({t1 - t0:.0f} ms)")
